# CausalRank — 联合 Bootstrap 协方差与不确定性量化（Phase 2）

本 Notebook 演示 Phase 2 的统计推断内核：

1. **联合重抽样**：每轮对同一行索引有放回重抽样（一个索引向量服务全部 Treatment）；
2. **每轮重拟合全部协议**：插补/编码/GBM/折划分/权重/K 组均值全部重跑；
3. 协方差 $\hat\Sigma = \mathrm{np.cov}(draws, rowvar=False, ddof=1)$，
   **绝不**再除以 $n$（每行是完整估计量，不是单客户贡献值）；
4. 单项 CI：正态近似 $\hat\mu_a \pm z_{1-\alpha/2}\sqrt{\hat\Sigma_{aa}}$；
5. 失败轮（稀有臂被漏抽等）被记录并披露；成功率过低时**阻断推断**；
6. Monte Carlo 校准：bootstrap SE 与真实抽样波动的比较。

> **排名推断属于 Phase 3**：本 Notebook 不做多重比较校正的两两区间或排名置信集，
> 点估计排序仍不是统计上确立的排名。不确定性只覆盖估计量的抽样波动，
> 不覆盖识别假设被违背带来的偏差。

In [1]:
import numpy as np
import pandas as pd

from causalrank import CausalRank
from causalrank.simulation import make_dataset

pd.set_option("display.width", 130)

## 1. 启用联合 Bootstrap 的完整分析

`n_bootstrap=120` 是显式开启推断的开关（默认 `None` 只给点估计，避免无意中跑几分钟）。
研究级配置建议 `n_bootstrap=300` 以上；这里为 Notebook 执行时间取小预算。

In [2]:
ds = make_dataset("confounded", n=2000, random_state=2026)
model = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    cross_fit=3,
    n_bootstrap=120,
    ci_method="normal",
    n_jobs=1,
    random_state=2026,
)
result = model.fit(ds.df)
print("inference_available:", result.inference_available)
display(result.effects.to_dataframe().round(4))
print({k: v for k, v in result.metadata["bootstrap"].items() if k != "failure_reasons"})

inference_available: True


,label,n_treated,raw_mean,mu,se,ci_low,ci_high,weight_sum,ess
0,A,754,0.4549,0.4219,0.0174,0.3878,0.4559,2231.1808,584.4989
1,B,646,0.4582,0.5412,0.0304,0.4816,0.6009,2303.3496,252.5057
2,C,600,0.4350,0.3834,0.0211,0.3421,0.4247,2297.7461,394.6472


{'n_requested': 120, 'n_success': 120, 'n_failed': 0, 'failure_rate': 0.0, 'base_seed': 2026, 'usable': True}


In [3]:
est = result.effects.to_dataframe().set_index("label")
check = pd.DataFrame(
    {
        "mu_hat": est["mu"],
        "se": est["se"],
        "ci_low": est["ci_low"],
        "ci_high": est["ci_high"],
        "mu_true": ds.mu_true,
        "covered": (ds.mu_true >= est["ci_low"]) & (ds.mu_true <= est["ci_high"]),
    }
)
display(check.round(4))

,mu_hat,se,ci_low,ci_high,mu_true,covered
A,0.4219,0.0174,0.3878,0.4559,0.4367,True
B,0.5412,0.0304,0.4816,0.6009,0.5185,True
C,0.3834,0.0211,0.3421,0.4247,0.4024,True


## 2. 协方差矩阵：联合估计的依赖结构

非对角元素来自**同一次**重抽样下的联合估计——两臂共享同一批被抽中的客户，
因此 $\hat\Sigma_{ij}$ 携带真实的估计相关性。检验相关矩阵与 PSD：

In [4]:
sigma = result.covariance
display(pd.DataFrame(sigma, index=est.index, columns=est.index).round(6))
d = np.sqrt(np.diag(sigma))
corr = sigma / np.outer(d, d)
display(pd.DataFrame(corr, index=est.index, columns=est.index).round(3))
print("min eigenvalue:", float(np.linalg.eigvalsh(sigma).min()))
print("SE == sqrt(diag(Sigma)):", np.allclose(est["se"], d, rtol=1e-10))

label,A,B,C
label,,,
A,0.000302,-0.000044,0.000012
B,-0.000044,0.000925,0.000028
C,0.000012,0.000028,0.000444


label,A,B,C
label,,,
A,1.000,-0.084,0.034
B,-0.084,1.000,0.043
C,0.034,0.043,1.000


min eigenvalue: 0.0002972559499534128
SE == sqrt(diag(Sigma)): True


## 3. 可复现性：同种子逐位一致，且与执行顺序无关

每轮种子由 `SeedSequence(base_seed).spawn(B)` 派生，同时驱动重抽样与模型随机性；
因此 `n_jobs=2` 并行与单线程结果逐位相同。

In [5]:
small = {"n_bootstrap": 12, "bootstrap_min_successes": 8}
a = CausalRank(treatment="treatment", outcome="outcome", confounders=["x1", "x2", "x3", "x4"],
               random_state=7, **small).fit(ds.df)
b = CausalRank(treatment="treatment", outcome="outcome", confounders=["x1", "x2", "x3", "x4"],
               random_state=7, **small).fit(ds.df)
c = CausalRank(treatment="treatment", outcome="outcome", confounders=["x1", "x2", "x3", "x4"],
               random_state=7, n_jobs=2, **small).fit(ds.df)
print("same seed identical   :", np.array_equal(a.covariance, b.covariance))
print("n_jobs=2 identical    :", np.array_equal(a.covariance, c.covariance))

same seed identical   : True
n_jobs=2 identical    : True


## 4. 失败轮与推断阻断：稀有 Treatment 的诚实处理

当某轮重抽样漏掉稀有臂时，该轮**无法**给出完整 K 维向量：记录失败原因并排除，
绝不静默丢弃；成功率低于门槛（默认 90% 且至少 30 轮）时**阻断推断**。

In [6]:
rare = make_dataset("rare_treatment", n=70, random_state=3)
rare_result = CausalRank(
    treatment="treatment",
    outcome="outcome",
    confounders=["x1", "x2", "x3", "x4"],
    n_bootstrap=40,
    bootstrap_min_successes=20,
    bootstrap_min_success_rate=1.0,  # 任何失败都阻断
    random_state=11,
).fit(rare.df)
print("inference_available:", rare_result.inference_available)
print("成功/失败轮:", rare_result.metadata["bootstrap"]["n_success"],
      "/", rare_result.metadata["bootstrap"]["n_failed"])
for w in rare_result.warnings:
    if "bootstrap" in w.code:
        print(f"{w.level:>8} | {w.code:<30} | {w.message[:110]}")

inference_available: False
成功/失败轮: 35 / 5
 warning | bootstrap_failures             | 5/40 bootstrap rounds failed (rate 12.5%) and were excluded from the draw matrix; failure reasons: {'CausalRan
critical | bootstrap_inference_blocked    | Joint bootstrap inference is BLOCKED: 35/40 rounds succeeded (required: success rate >= 100% and >= 20 success


## 5. Monte Carlo 校准：SE 是否对得上真实抽样波动？

`se_ratio = 平均 bootstrap SE / mu_hat 的经验标准差`；覆盖率同时报告 MC 误差
$\sqrt{p(1-p)/R}$。研究级脚本：`scripts/run_se_validation.py`（R=24、B=150 的结果
见 `docs/agent_progress.md`）。下面是一个小预算演示：

In [7]:
from causalrank.simulation import run_se_benchmark

se_res = run_se_benchmark("confounded", n=600, n_replications=6, n_bootstrap=30, random_state=5)
display(se_res.summary.round(3))
print("注意：R=6 的 MC 误差很大（coverage 的 MC se 约 0.10），只演示流程不下结论")

,scenario,n,label,n_replications,mu_true,bias,empirical_sd,mean_se,se_ratio,coverage,coverage_mc_se,ci_method,n_bootstrap
0,confounded,600,A,6,0.437,-0.000,0.040,0.033,0.816,0.833,0.152,normal,30
1,confounded,600,B,6,0.518,-0.047,0.032,0.035,1.090,0.500,0.204,normal,30
2,confounded,600,C,6,0.402,0.014,0.016,0.038,2.389,1.000,0.000,normal,30


注意：R=6 的 MC 误差很大（coverage 的 MC se 约 0.10），只演示流程不下结论


## 6. 边界与下一步

1. CI 覆盖的是**估计量在拟合模型下的抽样波动**，不覆盖未观测混杂等识别假设违背带来的偏差；
2. 重新拟合的非参数 bootstrap + ML nuisance 的理论性质依赖算法稳定性，
   必须用模拟覆盖率检验（上节）而不是当作通用推断保证；
3. 单项 CI **不能**代替两两显著性检验；`ci_method="percentile"` 可选，
   但其适用条件是 bootstrap 分布能近似抽样分布；
4. **Phase 3**：Bonferroni 同时差值区间、可确认优劣关系、保守排名置信集。

定义与协议见 `docs/methodology.md` 与 `docs/simulation_protocol.md`。